# listm'

In [ ]:
///- --test static

open testing

## listm'

### append

In [ ]:
instance append list t =
    listm.append

In [1]:
///- --test --print-code
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

[ "a"; "b" ] ++ [ "c"; "d" ]
|> _assert_eq [ "a"; "b"; "c"; "d" ]

.gleam (gleam)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Cons(d, Nil)))) }
.fsx (fsharp)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Cons(d, Nil)))) }
.py (cuda)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Cons(d, Nil)))) }
.rs (rust)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Cons(d, Nil)))) }
.ts (typescript)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Cons(d, Nil)))) }

0

### collect

In [ ]:
inl collect forall t r. (fn : t -> list r) (items : list t) : list r =
    items
    |> listm.map fn
    |> listm.fold (++) []

### replicate

In [ ]:
inl replicate count x =
    listm.init count fun _ => x

### map4

In [ ]:
inl map4 f l1 l2 l3 l4 =
    inl rec 루프 l1 l2 l3 l4 acc =
        match l1, l2, l3, l4 with
        | (x1 :: xs1), (x2 :: xs2), (x3 :: xs3), (x4 :: xs4) =>
            루프 xs1 xs2 xs3 xs4 (f x1 x2 x3 x4 :: acc)
        | _ => acc |> listm.rev
    루프 l1 l2 l3 l4 []

### init_series

In [ ]:
inl init_series start end inc =
    inl total : f64 = conv ((end - start) / inc) + 1
    listm.init total (conv >> (*) inc >> (+) start)

In [2]:
///- --test

init_series 0 1 0.5
|> _assert_eq [ 0f64; 0.5; 1 ]

__assert_eq { expected = Cons(0.0, Cons(0.5, Cons(1.0, Nil))) }

### try_item

In [ ]:
inl rec try_item i = function
    | Cons (x, _) when i = 0 => Some x
    | Cons (_, xs) => try_item (i - 1) xs
    | Nil => None

In [3]:
///- --test

listm.init 10i32 id
|> try_item 9i32
|> _assert_eq (Some 9)

__assert_eq { expected = Some(9) }

In [4]:
///- --test

listm.init 10i32 id
|> try_item 10i32
|> _assert_eq None

__assert_eq { expected = None }

### item

In [ ]:
inl item i =
    try_item i >> optionm.value

In [5]:
///- --test

listm.init 10i32 id
|> item 9i32
|> _assert_eq 9

__assert_eq { expected = 9 }

In [6]:
///- --test
///> fsharp

fun () =>
    listm.init 10i32 id
    |> item 10i32
    |> ignore
|> _throws
|> optionm.map sm'.format_exception
|> _assert_eq (Some "System.Exception: Option does not have a value.")

__assert_eq { expected = Some(System.Exception: Option does not have a value.) }

### try_item_

In [ ]:
let rec try_item_ i = function
    | Cons (x, _) when i = 0 => Some x
    | Cons (_, xs) => try_item_ (i - 1) xs
    | Nil => None

### item_

In [ ]:
inl item_ i =
    try_item_ i >> optionm.value

### sum

In [ ]:
inl sum list =
    list |> listm.fold (+) 0

In [7]:
///- --test

listm.init 10i32 id
|> sum
|> _assert_eq 45

__assert_eq { expected = 45 }

### unzip

In [ ]:
inl unzip list =
    (([], []), list)
    ||> listm.fold fun (acc_x, acc_y) (x, y) =>
        x :: acc_x, y :: acc_y
    |> fun x, y =>
        x |> listm.rev, y |> listm.rev

In [8]:
///- --test

listm.init 10i32 id
|> listm.map (fun x => x, x)
|> unzip
|> fun x, y =>
    x |> sum
    |> _assert_eq 45

    y |> sum
    |> _assert_eq 45

__assert_eq { expected = 45 }
__assert_eq { expected = 45 }

### try_index_of

In [ ]:
inl try_index_of forall b c {number}. (item : b) (list : list b) : option c =
    inl rec 루프 forall a {number}. (i : a) : list b -> option a = function
        | [] => None
        | x :: xs =>
            if x = item
            then Some i
            else 루프 (i + 1) xs
    루프 0 list

inl index_of item =
    try_index_of item >> optionm.value

inl try_index_of_ forall b c {number}. (item : b) (list : list b) : option c =
    let rec 루프 forall a {number}. (i : a) : list b -> option a = function
        | [] => None
        | x :: xs =>
            if x = item
            then Some i
            else 루프 (i + 1) xs
    루프 0 list

inl index_of_ item =
    try_index_of_ item >> optionm.value

inl try_index_of__ item list =
    inl i = mut 0
    inl list = mut list
    inl result = mut None
    let rec 루프 () =
        match *list with
        | [] => result <- None
        | x :: xs =>
            if x = item
            then result <- Some *i
            else
                i <- *i + 1
                list <- xs
                루프 ()
    루프 ()
    *result

inl index_of__ item =
    try_index_of__ item >> optionm.value

In [9]:
///- --test

listm.init 10i32 id
|> index_of 5i32
|> _assert_eq 5i32

__assert_eq { expected = 5 }

In [10]:
///- --test

listm.init 10i32 id
|> try_index_of 10i32
|> _assert_eq (None : option i32)

__assert_eq { expected = None }

### try_find

In [ ]:
inl try_find fn list =
    inl rec 루프 = function
        | [] => None
        | x :: xs =>
            if fn x
            then Some x
            else 루프 xs
    루프 list

inl try_find_ fn list =
    let rec 루프 = function
        | [] => None
        | x :: xs =>
            if fn x
            then Some x
            else 루프 xs
    루프 list

In [11]:
///- --test

listm.init 10i32 id
|> try_find ((=) 5i32)
|> _assert_eq (Some 5i32)

__assert_eq { expected = Some(5) }

In [ ]:
inl find x =
    try_find x >> optionm.value

inl find_ x =
    try_find_ x >> optionm.value

In [12]:
///- --test

listm.init 10i32 id
|> find ((=) 5i32)
|> _assert_eq 5i32

__assert_eq { expected = 5 }

### choose

In [ ]:
inl choose f l =
    (l, [])
    ||> listm.foldBack fun x acc =>
        match f x with
        | Some y => y :: acc
        | None => acc

In [13]:
///- --test

listm.init 10i32 id
|> choose (fun x => if x % 2 = 0 then Some x else None)
|> _assert_eq [ 0; 2; 4; 6; 8 ]

__assert_eq { expected = Cons(0, Cons(2, Cons(4, Cons(6, Cons(8, Nil))))) }

### filter

In [ ]:
inl filter forall t. (fn : t -> bool) (list : list t) : list t =
    (list, Nil)
    ||> listm.foldBack fun x acc =>
        if fn x then x :: acc else acc

### zip_with

In [ ]:
inl zip_with fn xs ys =
    inl rec 루프 acc xs ys =
        match xs, ys with
        | Cons (x, xs), Cons (y, ys) =>
            루프 (fn x y :: acc) xs ys
        | _ => listm.rev acc
    루프 [] xs ys

inl zip_with_ fn xs ys =
    let rec 루프 acc xs ys =
        match xs, ys with
        | Cons (x, xs), Cons (y, ys) =>
            루프 (fn x y :: acc) xs ys
        | _ => listm.rev acc
    루프 [] xs ys

In [14]:
///- --test

([ 1i32; 2; 3 ], [ 4; 5; 6 ])
||> zip_with (+)
|> _assert_eq [ 5; 7; 9 ]

__assert_eq { expected = Cons(5, Cons(7, Cons(9, Nil))) }

### zip

In [ ]:
inl zip xs ys =
    zip_with pair xs ys

inl zip_ xs ys =
    zip_with_ pair xs ys

In [15]:
///- --test

([ 1i32; 2; 3 ], [ 4i32; 5; 6 ])
||> zip
|> _assert_eq [ 1, 4; 2, 5; 3, 6 ]

__assert_eq { expected = Cons((1, 4), Cons((2, 5), Cons((3, 6), Nil))) }

### indexed

In [ ]:
inl indexed list =
    (([], 0), list)
    ||> listm.fold fun (acc, i) x =>
        (i, x) :: acc, i + 1
    |> fst
    |> listm.rev

In [16]:
///- --test

listm.init 5i32 ((*) 2)
|> indexed
|> _assert_eq [ 0i32, 0; 1, 2; 2, 4; 3, 6; 4, 8 ]

__assert_eq { expected = Cons((0, 0), Cons((1, 2), Cons((2, 4), Cons((3, 6), Cons((4, 8), Nil))))) }

### group_by

In [ ]:
inl group_by fn list =
    (list, [])
    ||> listm.foldBack fun x acc =>
        inl xk = fn x
        inl found, new_acc =
            ((false, []), acc)
            ||> listm.fold fun (found, acc') (k, xs) =>
                if k = xk
                then true, (k, x :: xs) :: acc'
                else found, (k, xs) :: acc'
        if found
        then new_acc
        else (xk, [ x ]) :: new_acc

In [17]:
///- --test

listm.init 10i32 id
|> group_by (fun x => x % 2 = 0)
|> _assert_eq [ true, [ 0; 2; 4; 6; 8 ]; false, [ 1; 3; 5; 7; 9 ] ]

__assert_eq { expected = Cons((true, Cons(0, Cons(2, Cons(4, Cons(6, Cons(8, Nil)))))), Cons((false, Cons(1, Cons(3, Cons(5, Cons(7, Cons(9, Nil)))))), Nil)) }

### forall'

In [ ]:
inl forall' fn (head :: tail) =
    (true, tail)
    ||> listm.fold fun acc x =>
        acc && x = head

In [18]:
///- --test

[ 1i32; 1; 1; 1; 1 ]
|> forall' ((=) 1i32)
|> _assert_eq true

__assert_eq { expected = true }

### last

In [ ]:
inl last list =
    list
    |> listm.rev
    |> item 0i32

In [19]:
///- --test

listm.init 10i32 id
|> last
|> _assert_eq 9

__assert_eq { expected = 9 }

### try_pick

In [ ]:
inl try_pick fn list =
    inl rec body fn = function
        | [] => None
        | x :: xs =>
            match fn x with
            | Some y => Some y
            | None => 루프 xs
    and inl 루프 list =
        if var_is list |> not
        then body fn list
        else
            inl fn = join fn
            inl list = dyn list
            join body fn list
    루프 list

In [20]:
///- --test

listm.init 10i32 id
|> try_pick (fun x => if x = 5i32 then Some x else None)
|> _assert_eq (Some 5i32)

__assert_eq { expected = Some(5) }

### exists'

In [ ]:
inl exists' f x =
    inl rec body f = function
        | [] => false
        | x :: xs =>
            if f x
            then true
            else 루프 xs
    and inl 루프 list =
        if var_is list |> not
        then body f list
        else
            inl f = join f
            inl list = dyn list
            join body f list
    루프 x

In [21]:
///- --test --print-code
///> gleam
///> fsharp
///> cuda
///> typescript
///> cuda

[ 'a'; 'b'; 'c' ]
|> exists' fun x => x = 'b'
|> _assert_eq true

[ 'a'; 'b' ]
|> exists' fun x => x = 'c'
|> _assert_eq false

[]
|> exists' fun x => x = 'a'
|> _assert_eq false

.gleam (gleam)
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = false }
.fsx (fsharp)
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = false }
.py (cuda)
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = false }
.ts (typescript)
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = false }
.py (cuda)
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = false }

## fsharp

### list'

In [ ]:
inl list'_types () =
    backend_switch {
        Gleam = fun () => global "import gleam/list"
        Rust = fun () => ()
    }
type list'_switch t =
    {
        Gleam : $'List(`t)'
        Fsharp : $'`t list'
        Python : $'list'
        Rust : $'Rc<Vec<`t>>'
        TypeScript : $'Array<`t>'
        Zig : $'[]const `(t)'
    }
nominal list' t = $'backend_switch `(list'_switch t)'

### empty'

In [ ]:
inl empty' forall t. () : list' t =
    backend_switch {
        Gleam = fun () => $'[]' : list' t
        Lua = fun () => $'{}' : list' t
        Fsharp = fun () => $'[]' : list' t
        Python = fun () => $'[]' : list' t
        Rust = fun () => $'Rc::new(<Vec<`t>>::new())' : list' t
        TypeScript = fun () => $'[]' : list' t
        Zig = fun () => $'&[_]`t{}' : list' t
    }

### cons'

In [ ]:
inl cons' forall t. (head : t) (tail : list' t) : list' t =
    backend_switch {
        Gleam = fun () => $'[ !head, ..!tail ]' : list' t
        Lua = fun () =>
            $'table.insert(!tail, 1, !head)'
            $'!tail ' : list' t
        Fsharp = fun () => $'!head :: !tail ' : list' t
        Python = fun () =>
            $'!tail.insert(0, !head)'
            $'!tail ' : list' t
        Rust = fun () => $'Rc::new({ let mut v = (!tail).as_ref().clone(); v.insert(0, !head); v })' : list' t
        TypeScript = fun () => $'[!head, ...!tail]' : list' t
        Zig = fun () =>
            inl len : i32 = $'\@as(i32, \@intCast(!tail.len))'
            inl r : a i32 t = am.init (len + 1i32) fun i => if i = 0i32 then head else $'!tail[\@intCast(!i - 1)]' : t
            inl (a r) = r
            $'!r ' : list' t
    }

### box

In [ ]:
inl box forall t. (list : list t) : list' t =
    run_target function
        | Zig => fun () =>
            (list, empty' ()) ||> listm.foldBack cons'
        | Rust => fun () =>
            inl acc : $'Rc<RefCell<Vec<`t>>>' = $'Rc::new(RefCell::new(Vec::new()))'
            inl _ : i32 = (0i32, list) ||> listm.fold fun n (x : t) =>
                $'!acc.borrow_mut().push(!x)' : ()
                n + 1
            $'Rc::new(!acc.borrow().clone())' : list' t
        | _ => fun () => (list, empty' ()) ||> listm.foldBack cons'

### rev'

In [ ]:
inl rev' forall t. (items : list' t) : list' t =
    backend_switch {
        Gleam = fun () => $'!items |> list.reverse' : list' t
        Lua = fun () =>
            inl result : mut (list' t) = empty' () |> mut
            inl len = $'##(!items)' : i32
            let rec 루프 i =
                if i < 0i32
                then *result
                else
                    result <- cons' ($'!items[!i + 1]') (*result)
                    루프 (i - 1i32)
            루프 (len - 1i32)
        Fsharp = fun () => items |> $'List.rev' : list' t
        Python = fun () => $'list(reversed(!items))' : list' t
        Rust = fun () => $'Rc::new({ let mut v = (!items).as_ref().clone(); v.reverse(); v })' : list' t
        TypeScript = fun () => $'[...!items].reverse()' : list' t
        Zig = fun () => $'blk: { const r = spiral_gpa.dupe(`t, !items) catch \@panic("out of memory"); std.mem.reverse(`t, r); break :blk r; }' : list' t
    }

### fold'

In [ ]:
inl fold' forall t u. (fn : t -> u) (init : list u) (list : list' t) : list u =
    backend_switch {
        Gleam = fun () =>
            $'let listm_fold = !list |> list.fold (!init, fn (acc, x) {'
            $'!(Cons (fn $'x', $'acc'))' : ()
            $'})' : ()
            $'listm_fold' : list u
        Lua = fun () =>
            inl result : mut (list u) = init |> mut
            inl len = $'##(!list)' : i32
            let rec 루프 i =
                if i < 0i32
                then *result
                else
                    result <- $'!fn(!list[!i + 1])' :: *result
                    루프 (i - 1i32)
            루프 (len - 1i32)
        Fsharp = fun () =>
            (init, list)
            ||> $'List.fold' join fun acc x => Cons (fn x, acc)
            : list u
        Python = fun () =>
            inl init = init |> box
            $'r = !init '
            inl list = list |> rev'
            $'for x in !list: r = [!fn(x)] + r'
            inl init : list u = Nil
            inl cons (a : u) b = Cons (a, b)
            $'r_ = !init '
            $'for x in r: r_ = !cons (x)(r_)'
            $'r_' : list u
        Rust = fun () =>
            inl len : i32 = $'(!list).len() as i32'
            let rec 루프 (i : i32) (acc : list u) : list u =
                if i >= len then acc
                else
                    inl item : t = $'(!list)[!i as usize].clone()'
                    루프 (i + 1i32) (Cons (fn item, acc))
            루프 0i32 init
        TypeScript = fun () =>
            inl len : i32 = $'!list.length'
            let rec 루프 (i : i32) (acc : list u) : list u =
                if i >= len then acc
                else
                    inl item : t = $'!list[!i]'
                    루프 (i + 1i32) (Cons (fn item, acc))
            루프 0i32 init
        Zig = fun () =>
            inl len : i32 = $'\@as(i32, \@intCast(!list.len))'
            let rec 루프 (i : i32) (acc : list u) : list u =
                if i >= len then acc
                else
                    inl item : t = $'!list[\@intCast(!i)]'
                    루프 (i + 1i32) (Cons (fn item, acc))
            루프 0i32 init
    }

### fold_back'

In [ ]:
inl fold_back' forall t u. (fn : t -> u) (list : list' t) (init : list u) : list u =
    backend_switch {
        Gleam = fun () =>
            global "import gleam/list"
            $'let listm_fold_back = !list |> list.fold_right (!init, fn (acc, x) {'
            $'!(Cons (fn $'x', $'acc'))' : ()
            $'})' : ()
            $'listm_fold_back' : list u
        Lua = fun () =>
            list
            |> rev'
            |> fold' fn init
        Fsharp = fun () =>
            (list, init)
            ||> $'List.foldBack' join fun x acc => Cons (fn x, acc)
            : list u
        Python = fun () =>
            list
            |> rev'
            |> fold' fn init
        Rust = fun () =>
            inl len : i32 = $'(!list).len() as i32'
            let rec 루프 (i : i32) (acc : list u) : list u =
                if i < 0i32 then acc
                else
                    inl item : t = $'(!list)[!i as usize].clone()'
                    루프 (i - 1i32) (Cons (fn item, acc))
            루프 (len - 1i32) init
        TypeScript = fun () =>
            inl len : i32 = $'!list.length'
            let rec 루프 (i : i32) (acc : list u) : list u =
                if i < 0i32 then acc
                else
                    inl item : t = $'!list[!i]'
                    루프 (i - 1i32) (Cons (fn item, acc))
            루프 (len - 1i32) init
        Zig = fun () =>
            inl len : i32 = $'\@as(i32, \@intCast(!list.len))'
            let rec 루프 (i : i32) (acc : list u) : list u =
                if i < 0i32 then acc
                else
                    inl item : t = $'!list[\@intCast(!i)]'
                    루프 (i - 1i32) (Cons (fn item, acc))
            루프 (len - 1i32) init
    }

### filter'

In [ ]:
inl filter' forall t. (fn : t -> bool) (list : list' t) : list' t =
    backend_switch {
        Gleam = fun () => $'!list |> list.filter(!fn)' : list' t
        Lua = fun () =>
            inl result : mut (list' t) = empty' () |> mut
            inl len = $'##(!list)' : i32
            let rec 루프 i =
                if i >= len
                then *result
                else
                    inl item = $'!list[!i + 1]' : t
                    if fn item
                    then
                        result <- cons' item (*result)
                        루프 (i + 1i32)
                    else 루프 (i + 1i32)
            루프 0i32
        Fsharp = fun () => list |> $'"List.filter !fn"' : list' t
        Python = fun () => $'list(filter(!fn, !list))' : list' t
        Rust = fun () =>
            inl len : i32 = $'(!list).len() as i32'
            let rec 루프 (i : i32) (acc : list t) : list t =
                if i < 0i32 then acc
                else
                    inl item : t = $'(!list)[!i as usize].clone()'
                    루프 (i - 1i32) (if fn item then Cons (item, acc) else acc)
            루프 (len - 1i32) Nil |> box
        TypeScript = fun () =>
            inl len : i32 = $'!list.length'
            let rec 루프 (i : i32) (acc : list t) : list t =
                if i < 0i32 then acc
                else
                    inl item : t = $'!list[!i]'
                    루프 (i - 1i32) (if fn item then Cons (item, acc) else acc)
            루프 (len - 1i32) Nil |> box
        Zig = fun () =>
            inl len : i32 = $'\@as(i32, \@intCast(!list.len))'
            let rec 루프 (i : i32) (acc : list t) : list t =
                if i < 0i32 then acc
                else
                    inl item : t = $'!list[\@intCast(!i)]'
                    루프 (i - 1i32) (if fn item then Cons (item, acc) else acc)
            루프 (len - 1i32) Nil |> box
    }

### map

In [ ]:
inl map forall t u. (fn : t -> u) (list : list' t) : list' u =
    list'_types ()
    backend_switch {
        Gleam = fun () => $'!list |> list.map(!fn)' : list' u
        Lua = fun () =>
            inl result : mut (list' u) = empty' () |> mut
            inl len = $'##(!list)' : i32
            let rec 루프 i =
                if i >= len
                then *result
                else
                    result <- cons' ($'!fn(!list[!i + 1])') (*result)
                    루프 (i + 1i32)
            루프 0i32
        Fsharp = fun () => list |> $'List.map' fn : list' u
        Python = fun () => $'list(map(!fn, !list))' : list' u
        Rust = fun () => fold_back' fn list Nil |> box
        TypeScript = fun () => fold_back' fn list Nil |> box
        Zig = fun () => fold_back' fn list Nil |> box
    }

### unbox

In [ ]:
inl unbox forall t. (list : list' t) : list t =
    (list, Nil)
    ||> fold_back' id

### distinct'

In [ ]:
inl distinct' forall t. (list : list' t) : list' t =
    list'_types ()
    backend_switch {
        Gleam = fun () => $'!list |> list.unique' : list' t
        Lua = fun () =>
            inl result : mut (list' t) = empty' () |> mut
            inl len = $'##(!list)' : i32
            inl seen : mut (list t) = [] |> mut
            let rec 루프 i =
                if i >= len
                then *result
                else
                    inl item = $'!list[!i + 1]' : t
                    if exists' (fun x => x = item) (*seen)
                    then 루프 (i + 1i32)
                    else
                        seen <- item :: *seen
                        result <- cons' item (*result)
                        루프 (i + 1i32)
            루프 0i32
        Fsharp = fun () => list |> $'List.distinct' : list' t
        Python = fun () =>
            $'x = list(set(!list))'
            $'x.sort(key=!list.index)'
            $'x' : list' t
        Rust = fun () =>
            inl len : i32 = $'(!list).len() as i32'
            let rec 루프 (i : i32) (seen : list t) : list t =
                if i >= len then seen
                else
                    inl item : t = $'(!list)[!i as usize].clone()'
                    루프 (i + 1i32) (if exists' (fun x => x = item) seen then seen else Cons (item, seen))
            루프 0i32 Nil |> box |> rev'
        TypeScript = fun () =>
            inl len : i32 = $'!list.length'
            let rec 루프 (i : i32) (seen : list t) : list t =
                if i >= len then seen
                else
                    inl item : t = $'!list[!i]'
                    루프 (i + 1i32) (if exists' (fun x => x = item) seen then seen else Cons (item, seen))
            루프 0i32 Nil |> box |> rev'
        Zig = fun () =>
            inl len : i32 = $'\@as(i32, \@intCast(!list.len))'
            let rec 루프 (i : i32) (seen : list t) : list t =
                if i >= len then seen
                else
                    inl item : t = $'!list[\@intCast(!i)]'
                    루프 (i + 1i32) (if exists' (fun x => x = item) seen then seen else Cons (item, seen))
            루프 0i32 Nil |> box |> rev'
    }

In [22]:
///- --test
///> gleam
///> fsharp
///> rust
///> cuda
///> typescript

[ "1"; "2"; "2"; "3" ]
|> box
|> distinct'
|> unbox
|> _assert_eq [ "1"; "2"; "3" ]

.gleam (gleam)
__assert_eq { expected = Cons(1, Cons(2, Cons(3, Nil))) }
.fsx (fsharp)
__assert_eq { expected = Cons(1, Cons(2, Cons(3, Nil))) }
.rs (rust)
__assert_eq { expected = Cons(1, Cons(2, Cons(3, Nil))) }
.py (cuda)
__assert_eq { expected = Cons(1, Cons(2, Cons(3, Nil))) }
.ts (typescript)
__assert_eq { expected = Cons(1, Cons(2, Cons(3, Nil))) }

0

### to_array'

In [ ]:
inl to_array' forall t. (items : list' t) : array_base t =
    backend_switch {
        Gleam = fun () => $'!items |> spiral_array_from_list' : array_base t
        Lua = fun () =>
            inl len = $'##(!items)' : i32
            inl d = real defaultof `t ()
            inl result : a _ t = am.init len (fun _ => d)
            let rec 루프 i =
                if i >= len then
                    inl (a result) = result
                    result
                else
                    set result i $'!items[!i + 1]'
                    루프 (i + 1i32)
            루프 0i32
        Fsharp = fun () => items |> $'List.toArray' : array_base t
        Python = fun () =>
            inl len = $'len(!items)' : i32
            inl result : a _ t = a ($'__import__("numpy").empty(!len, dtype=object)' : array_base t)
            let rec 루프 i =
                if i >= len then
                    inl (a result) = result
                    result
                else
                    set result i $'!items[!i]'
                    루프 (i + 1i32)
            루프 0i32
        Rust = fun () => $'Rc::new(RefCell::new((!items).as_ref().clone()))' : array_base t
        TypeScript = fun () => $'[...!items]' : array_base t
        Zig = fun () => $'(spiral_gpa.dupe(`t, !items) catch \@panic("out of memory"))' : array_base t
    }

### contains

In [ ]:
inl contains forall t. (item : t) (list : list t) : bool =
    list
    |> try_index_of_ item
    |> fun x => x : _ int
    |> (<>) None

### pairwise

In [23]:
inl pairwise forall t. (list : list t) : list (t * t) =
    let rec 루프 = function
        | (x1 : t) :: x2 :: xs => (x1, x2) :: 루프 (x2 :: xs)
        | _ => []
    루프 list